# ESM-1v Classification Experiments

This notebook runs all ESM-1v embedding experiments: four primary experiments
(mean pool and lysine-position embeddings × AcetylBench and DeepACET training
data), combined traditional feature baselines, a PCA dimensionality sensitivity
analysis, and the asymmetric CPLM→dbPTM cross-database evaluation.

Pre-computed ESM-1v embeddings are downloaded directly from Zenodo
(DOI: 10.5281/zenodo.19475597) — no GPU or embedding generation required
to reproduce the reported results.

**Output:** All benchmark metrics reported in the paper

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
    function clickConnect(){
        var btns = document.querySelectorAll("colab-toolbar-button#connect");
        if(btns.length > 0) btns[0].click();
    }
    setInterval(clickConnect, 60000);
'''))

import os, math, gc, pickle
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from itertools import product as iproduct
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
import os, math, gc
import urllib.request

emb_dir = '/content/esm1v_embeddings'
os.makedirs(emb_dir, exist_ok=True)

ZENODO_BASE = 'https://zenodo.org/records/21027250/files'

files = [
    'deepacet_pos_esm1v_mean.npy',
    'deepacet_neg_esm1v_mean.npy',
    'deepacet_pos_esm1v_kpos.npy',
    'deepacet_neg_esm1v_kpos.npy',
    'acetylbench_pos_esm1v_mean.npy',
    'acetylbench_neg_esm1v_mean.npy',
    'acetylbench_pos_esm1v_kpos.npy',
    'acetylbench_neg_esm1v_kpos.npy',
    'independent_test_pos_esm1v_mean.npy',
    'independent_test_neg_esm1v_mean.npy',
    'independent_test_pos_esm1v_kpos.npy',
    'independent_test_neg_esm1v_kpos.npy',
]

for fname in files:
    dest = os.path.join(emb_dir, fname)
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(
            f'{ZENODO_BASE}/{fname}?download=1', dest)
    else:
        print(f'  cached: {fname}')

print('All embeddings ready.')

In [ ]:
da_pos_mean = np.load(f'{emb_dir}/deepacet_pos_esm1v_mean.npy')
da_neg_mean = np.load(f'{emb_dir}/deepacet_neg_esm1v_mean.npy')
da_pos_kpos = np.load(f'{emb_dir}/deepacet_pos_esm1v_kpos.npy')
da_neg_kpos = np.load(f'{emb_dir}/deepacet_neg_esm1v_kpos.npy')

your_pos_mean = np.load(f'{emb_dir}/acetylbench_pos_esm1v_mean.npy')
your_neg_mean = np.load(f'{emb_dir}/acetylbench_neg_esm1v_mean.npy')
your_pos_kpos = np.load(f'{emb_dir}/acetylbench_pos_esm1v_kpos.npy')
your_neg_kpos = np.load(f'{emb_dir}/acetylbench_neg_esm1v_kpos.npy')

ind_pos_mean = np.load(f'{emb_dir}/independent_test_pos_esm1v_mean.npy')
ind_neg_mean = np.load(f'{emb_dir}/independent_test_neg_esm1v_mean.npy')
ind_pos_kpos = np.load(f'{emb_dir}/independent_test_pos_esm1v_kpos.npy')
ind_neg_kpos = np.load(f'{emb_dir}/independent_test_neg_esm1v_kpos.npy')

ind_all_mean = np.vstack([ind_pos_mean, ind_neg_mean])
ind_all_kpos = np.vstack([ind_pos_kpos, ind_neg_kpos])
ind_labels   = np.array([1]*len(ind_pos_mean) + [0]*len(ind_neg_mean))

print(f'AcetylBench pos mean: {your_pos_mean.shape}')
print(f'AcetylBench neg mean: {your_neg_mean.shape}')
print(f'DeepACET pos mean:    {da_pos_mean.shape}')
print(f'DeepACET neg mean:    {da_neg_mean.shape}')
print(f'Independent test:     {ind_all_mean.shape}')

In [ ]:
# PCA sensitivity analysis — ESM-1v embeddings
# Assesses whether uniform 128-dim reduction causes
# differential information loss across representation types

n_components_list = [50, 100, 128, 200, 256, 512]

print(f"{'Components':<12} {'ESM-1v Mean (%)':>18} {'ESM-1v K-pos (%)':>18}")
print("-" * 50)

for n in n_components_list:
    sc_tmp = StandardScaler()
    X_mean = sc_tmp.fit_transform(np.vstack([da_pos_mean, da_neg_mean]))
    pca_mean = PCA(n_components=n).fit(X_mean)
    var_mean = np.cumsum(pca_mean.explained_variance_ratio_)[-1] * 100

    X_kpos = sc_tmp.fit_transform(np.vstack([da_pos_kpos, da_neg_kpos]))
    pca_kpos = PCA(n_components=n).fit(X_kpos)
    var_kpos = np.cumsum(pca_kpos.explained_variance_ratio_)[-1] * 100

    marker = " <-- selected" if n == 128 else ""
    print(f"{n:<12} {var_mean:>17.2f}% {var_kpos:>17.2f}%{marker}")

del sc_tmp, X_mean, X_kpos, pca_mean, pca_kpos

In [ ]:
class AcetylDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


class MLP(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        h1 = min(256, input_dim)
        h2 = h1 // 2
        h3 = h2 // 2
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),        nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h2, h3),        nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h3, 1)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
def run_experiment(train_pos_emb, train_neg_emb, test_emb, test_labels,
                    n_pca, label, n_folds=5):
    from sklearn.model_selection import train_test_split

    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')

    train_emb    = np.vstack([train_pos_emb, train_neg_emb])
    train_labels = np.array([1]*len(train_pos_emb) + [0]*len(train_neg_emb))

    print(f'Training: {len(train_pos_emb)} pos + {len(train_neg_emb)} neg')
    print(f'Test:     {(test_labels==1).sum()} pos + {(test_labels==0).sum()} neg')

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_emb)
    pca = PCA(n_components=n_pca)
    train_pca = pca.fit_transform(train_scaled)
    test_pca  = pca.transform(scaler.transform(test_emb))

    cumvar = np.cumsum(pca.explained_variance_ratio_)[-1]
    print(f'PCA {n_pca} components: {cumvar*100:.1f}% variance')

    np.random.seed(42)
    pos_idx = np.where(train_labels == 1)[0]
    neg_idx = np.where(train_labels == 0)[0]
    n_min   = min(len(pos_idx), len(neg_idx))
    idx_bal = np.concatenate([
        np.random.choice(pos_idx, n_min, replace=False),
        np.random.choice(neg_idx, n_min, replace=False)])
    np.random.shuffle(idx_bal)
    X_bal = train_pca[idx_bal]
    y_bal = train_labels[idx_bal]
    print(f'Balanced: {n_min} pos + {n_min} neg')

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    cv_aucs = []
    print(f'\n--- 5-Fold Cross Validation ---')

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_bal, y_bal)):
        X_tr, X_val = X_bal[tr_idx], X_bal[val_idx]
        y_tr, y_val = y_bal[tr_idx], y_bal[val_idx]

        model = MLP(n_pca).to(device)
        lf    = nn.BCEWithLogitsLoss()
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
        best_auc, best_state, patience_count = 0, None, 0

        tr_dl = DataLoader(AcetylDataset(X_tr, y_tr), batch_size=64, shuffle=True)
        vl_dl = DataLoader(AcetylDataset(X_val, y_val), batch_size=64, shuffle=False)

        for epoch in range(150):
            model.train()
            for xb, yb in tr_dl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                lf(model(xb).squeeze(1), yb).backward()
                opt.step()

            model.eval()
            vl_loss, probs, trues = 0, [], []
            with torch.no_grad():
                for xb, yb in vl_dl:
                    xb, yb = xb.to(device), yb.to(device)
                    out = model(xb).squeeze(1)
                    vl_loss += lf(out, yb).item()
                    probs.extend(torch.sigmoid(out).cpu().tolist())
                    trues.extend(yb.cpu().tolist())
            auc = roc_auc_score(trues, probs)
            sch.step(vl_loss / len(vl_dl))
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= 15:
                    break

        model.load_state_dict(best_state)
        model.eval()
        probs, preds, trues = [], [], []
        with torch.no_grad():
            for xb, yb in vl_dl:
                xb = xb.to(device)
                p = torch.sigmoid(model(xb)).squeeze(1).cpu().tolist()
                probs.extend(p)
                preds.extend([1 if x >= 0.5 else 0 for x in p])
                trues.extend(yb.tolist())
        tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
        cv_aucs.append(roc_auc_score(trues, probs))
        print(f'  Fold {fold+1}: AUC={cv_aucs[-1]:.4f} Acc={accuracy_score(trues,preds):.4f} MCC={matthews_corrcoef(trues,preds):.4f}')

    cv_mean = np.mean(cv_aucs)
    cv_std  = np.std(cv_aucs)
    print(f'\nCV AUC: {cv_mean:.4f} ± {cv_std:.4f}')

    X_tr_f, X_vl_f, y_tr_f, y_vl_f = train_test_split(
        X_bal, y_bal, test_size=0.10, stratify=y_bal, random_state=42)
    print(f'Final train: {len(X_tr_f)}, validation: {len(X_vl_f)}')

    final = MLP(n_pca).to(device)
    lf    = nn.BCEWithLogitsLoss()
    opt   = torch.optim.Adam(final.parameters(), lr=1e-3, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    tr_dl = DataLoader(AcetylDataset(X_tr_f, y_tr_f), batch_size=64, shuffle=True)
    vl_dl = DataLoader(AcetylDataset(X_vl_f, y_vl_f), batch_size=64, shuffle=False)
    best_auc, best_state, patience_count = 0, None, 0

    for epoch in range(150):
        final.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            lf(final(xb).squeeze(1), yb).backward()
            opt.step()
        final.eval()
        vl_loss, probs, trues = 0, [], []
        with torch.no_grad():
            for xb, yb in vl_dl:
                xb, yb = xb.to(device), yb.to(device)
                out = final(xb).squeeze(1)
                vl_loss += lf(out, yb).item()
                probs.extend(torch.sigmoid(out).cpu().tolist())
                trues.extend(yb.cpu().tolist())
        auc = roc_auc_score(trues, probs)
        sch.step(vl_loss / len(vl_dl))
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Val AUC: {auc:.4f}')
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.clone() for k, v in final.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= 15:
                print(f'  Early stopping at epoch {epoch}, best val AUC: {best_auc:.4f}')
                break

    final.load_state_dict(best_state)
    final.eval()
    X_te = torch.tensor(test_pca, dtype=torch.float32)
    te_dl = DataLoader(torch.utils.data.TensorDataset(
        X_te, torch.tensor(test_labels, dtype=torch.float32)), batch_size=64, shuffle=False)

    probs, preds, trues = [], [], []
    with torch.no_grad():
        for xb, yb in te_dl:
            xb = xb.to(device)
            p = torch.sigmoid(final(xb)).squeeze(1).cpu().tolist()
            probs.extend(p)
            preds.extend([1 if x >= 0.5 else 0 for x in p])
            trues.extend(yb.tolist())

    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    ind = {'acc': accuracy_score(trues, preds),
           'auc': roc_auc_score(trues, probs),
           'mcc': matthews_corrcoef(trues, preds),
           'sn':  tp/(tp+fn), 'sp': tn/(tn+fp),
           'probs': probs, 'trues': trues, 'preds': preds}

    print(f'\nIndependent test AUC:  {ind["auc"]:.4f}')
    print(f'Independent test MCC:  {ind["mcc"]:.4f}')
    print(f'Sensitivity:           {ind["sn"]:.4f}')
    print(f'Specificity:           {ind["sp"]:.4f}')

    torch.save(final.state_dict(), os.path.join(emb_dir, f'{label.replace(" ","_")}_model.pth'))
    return cv_aucs, ind, final, scaler, pca, cv_mean

In [ ]:
cv1f, ind1f, m1f, sc1f, pc1f, cauc1f = run_experiment(
    da_pos_mean, da_neg_mean, ind_all_mean, ind_labels, 128,
    'ESM-1v Mean Pool — DeepACET Data')

In [ ]:
cv2f, ind2f, m2f, sc2f, pc2f, cauc2f = run_experiment(
    da_pos_kpos, da_neg_kpos, ind_all_kpos, ind_labels, 128,
    'ESM-1v Lysine-Position — DeepACET Data')

In [ ]:
cv3f, ind3f, m3f, sc3f, pc3f, cauc3f = run_experiment(
    your_pos_mean, your_neg_mean, ind_all_mean, ind_labels, 128,
    'ESM-1v Mean Pool — AcetylBench Data')

In [ ]:
cv4f, ind4f, m4f, sc4f, pc4f, cauc4f = run_experiment(
    your_pos_kpos, your_neg_kpos, ind_all_kpos, ind_labels, 128,
    'ESM-1v Lysine-Position — AcetylBench Data')

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'

def cksaap_encoding(sequence, k_max=3):
    seq = sequence.upper()
    features = []
    for k in range(k_max + 1):
        pair_counts = {a+b: 0 for a, b in iproduct(AA, AA)}
        total = 0
        for i in range(len(seq) - k - 1):
            pair = seq[i] + seq[i+k+1]
            if pair in pair_counts:
                pair_counts[pair] += 1
                total += 1
        features.extend([pair_counts[p]/total for p in pair_counts] if total > 0 else [0]*400)
    return features

BLOSUM62 = {
    'A':[4,-1,-2,-2,0,-1,-1,0,-2,-1,-1,-1,-1,-2,-1,1,0,-3,-2,0],
    'R':[-1,5,0,-2,-3,1,0,-2,0,-3,-2,2,-1,-3,-2,-1,-1,-3,-2,-3],
    'N':[-2,0,6,1,-3,0,0,0,1,-3,-3,0,-2,-3,-2,1,0,-4,-2,-3],
    'D':[-2,-2,1,6,-3,0,2,-1,-1,-3,-4,-1,-3,-3,-1,0,-1,-4,-3,-3],
    'C':[0,-3,-3,-3,9,-3,-4,-3,-3,-1,-1,-3,-1,-2,-3,-1,-1,-2,-2,-1],
    'Q':[-1,1,0,0,-3,5,2,-2,0,-3,-2,1,0,-3,-1,0,-1,-2,-1,-2],
    'E':[-1,0,0,2,-4,2,5,-2,0,-3,-3,1,-2,-3,-1,0,-1,-3,-2,-2],
    'G':[0,-2,0,-1,-3,-2,-2,6,-2,-4,-4,-2,-3,-3,-2,0,-2,-2,-3,-3],
    'H':[-2,0,1,-1,-3,0,0,-2,8,-3,-3,-1,-2,-1,-2,-1,-2,-2,2,-3],
    'I':[-1,-3,-3,-3,-1,-3,-3,-4,-3,4,2,-3,1,0,-3,-2,-1,-3,-1,3],
    'L':[-1,-2,-3,-4,-1,-2,-3,-4,-3,2,4,-2,2,0,-3,-2,-1,-2,-1,1],
    'K':[-1,2,0,-1,-3,1,1,-2,-1,-3,-2,5,-1,-3,-1,0,-1,-3,-2,-2],
    'M':[-1,-1,-2,-3,-1,0,-2,-3,-2,1,2,-1,5,0,-2,-1,-1,-1,-1,1],
    'F':[-2,-3,-3,-3,-2,-3,-3,-3,-1,0,0,-3,0,6,-4,-2,-2,1,3,-1],
    'P':[-1,-2,-2,-1,-3,-1,-1,-2,-2,-3,-3,-1,-2,-4,7,-1,-1,-4,-3,-2],
    'S':[1,-1,1,0,-1,0,0,0,-1,-2,-2,0,-1,-2,-1,4,1,-3,-2,-2],
    'T':[0,-1,0,-1,-1,-1,-1,-2,-2,-1,-1,-1,-1,-2,-1,1,5,-2,-2,0],
    'W':[-3,-3,-4,-4,-2,-2,-3,-2,-2,-3,-2,-3,-1,1,-4,-3,-2,11,2,-3],
    'Y':[-2,-2,-2,-3,-2,-1,-2,-3,2,-1,-1,-2,-1,3,-3,-2,-2,2,7,-1],
    'V':[0,-3,-3,-3,-1,-2,-2,-3,-3,3,1,-2,1,-1,-2,-2,0,-3,-1,4],
    'X':[0]*20,
}

def encode_blosum62(sequence, length=31):
    seq = sequence.upper().strip()[:length].ljust(length, 'X')
    return np.array([BLOSUM62.get(aa, BLOSUM62['X']) for aa in seq], dtype=np.float32).flatten()

def encode_onehot(sequence, length=31):
    seq = sequence.upper().strip()[:length].ljust(length, 'X')
    return np.array([1 if aa == a else 0 for aa in seq for a in AA], dtype=np.float32)

def encode_combined(sequence):
    return np.concatenate([cksaap_encoding(sequence), encode_blosum62(sequence), encode_onehot(sequence)])

In [ ]:
import urllib.request

GITHUB_BASE = 'https://raw.githubusercontent.com/uzairamu/AcetylBench/main'

for fname in ['distinct_positive_clean.csv', 'distinct_negative_clean.csv']:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(f'{GITHUB_BASE}/{fname}', fname)

pos_clean = pd.read_csv('distinct_positive_clean.csv')
neg_clean = pd.read_csv('distinct_negative_clean.csv')

print(f'AcetylBench pos: {len(pos_clean)}')
print(f'AcetylBench neg: {len(neg_clean)}')

`ind_pos` and `ind_neg` (independent test sequences) are sourced from
DeepAcet (Wu et al., 2019): https://github.com/Lab-Xu/DeepAcet

Download the independent test CSVs from that repository and load them
as `ind_pos` and `ind_neg` with a `sequence` column before running
the cell below.

print('Encoding combined features...')
pos_comb     = np.array([encode_combined(s) for s in pos_clean['sequence']])
neg_comb     = np.array([encode_combined(s) for s in neg_clean['sequence']])
ind_pos_comb = np.array([encode_combined(s) for s in ind_pos['sequence']])
ind_neg_comb = np.array([encode_combined(s) for s in ind_neg['sequence']])

X_train_c = np.vstack([pos_comb, neg_comb])
y_train_c = np.array([1]*len(pos_comb) + [0]*len(neg_comb))
X_test_c  = np.vstack([ind_pos_comb, ind_neg_comb])
y_test_c  = np.array([1]*len(ind_pos_comb) + [0]*len(ind_neg_comb))

np.random.seed(42)
n_min   = min((y_train_c==1).sum(), (y_train_c==0).sum())
idx_bal = np.concatenate([
    np.random.choice(np.where(y_train_c==1)[0], n_min, replace=False),
    np.random.choice(np.where(y_train_c==0)[0], n_min, replace=False)])
X_bal_c = X_train_c[idx_bal]
y_bal_c = y_train_c[idx_bal]

scaler_c  = StandardScaler()
X_bal_sc  = scaler_c.fit_transform(X_bal_c)
X_test_sc = scaler_c.transform(X_test_c)

lr_clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_clf.fit(X_bal_sc, y_bal_c)
lr_prob = lr_clf.predict_proba(X_test_sc)[:,1]
lr_pred = lr_clf.predict(X_test_sc)

rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_clf.fit(X_bal_sc, y_bal_c)
rf_prob = rf_clf.predict_proba(X_test_sc)[:,1]
rf_pred = rf_clf.predict(X_test_sc)

from sklearn.neural_network import MLPClassifier
mlp_clf = MLPClassifier(hidden_layer_sizes=(256,128), activation='relu',
                         max_iter=200, random_state=42,
                         early_stopping=True, validation_fraction=0.1)
mlp_clf.fit(X_bal_sc, y_bal_c)
mlp_prob = mlp_clf.predict_proba(X_test_sc)[:,1]
mlp_pred = mlp_clf.predict(X_test_sc)
y_ind    = y_test_c

print('\nCombined features on independent test:')
for name, prob, pred in [('LR', lr_prob, lr_pred), ('RF', rf_prob, rf_pred), ('MLP', mlp_prob, mlp_pred)]:
    tn, fp, fn, tp = confusion_matrix(y_ind, pred).ravel()
    print(f'  {name}: AUC={roc_auc_score(y_ind,prob):.4f} MCC={matthews_corrcoef(y_ind,pred):.4f} Sn={tp/(tp+fn):.4f} Sp={tn/(tn+fp):.4f}')

In [ ]:
your_all_kpos   = np.vstack([your_pos_kpos, your_neg_kpos])
your_labels_all = np.array([1]*len(your_pos_kpos) + [0]*len(your_neg_kpos))

your_pca = pc2f.transform(sc2f.transform(your_all_kpos))

m2f.eval()
your_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(your_pca, dtype=torch.float32)),
    batch_size=64, shuffle=False)

probs_your = []
with torch.no_grad():
    for (xb,) in your_loader:
        p = torch.sigmoid(m2f(xb.to(device))).squeeze(1).cpu().tolist()
        probs_your.extend(p)

preds_your = [1 if p >= 0.5 else 0 for p in probs_your]
tn, fp, fn, tp = confusion_matrix(your_labels_all, preds_your).ravel()

print('=== CPLM to dbPTM (asymmetric cross-database) ===')
print(f'AUC: {roc_auc_score(your_labels_all, probs_your):.4f}')
print(f'MCC: {matthews_corrcoef(your_labels_all, preds_your):.4f}')
print(f'Sn:  {tp/(tp+fn):.4f}')
print(f'Sp:  {tn/(tn+fp):.4f}')